In [1]:
# creating spark session
exec(open('/home/jovyan/.ipython/profile_
default/startup/00-spark-session.py').read())

Spark 3.5.0 session ready as `spark` (Delta Lake enabled).


In [2]:
# Adding Display functionality of Databricks 
exec(open('/home/jovyan/.ipython/profile_default/startup/01-databricks-utils.py').read())

Databricks-style helpers ready: display(), dbutils.fs/widgets/notebook, %run_notebook


In [3]:
from pyspark.sql.functions import *

## Problem Description

You are given two DataFrames:

- **`customers`**:
  - `customer_id` (int): Unique customer identifier
  - `name` (str): Customer name
  - `country` (str): Customer country

- **`orders`**:
  - `order_id` (int): Unique order identifier
  - `customer_id` (int): Customer who placed the order
  - `order_date` (str): Date of the order (YYYY‑MM‑DD)
  - `quantity` (int): Number of items ordered
  - `price` (int): Price per item

**Task:**  
Find customers who have placed **at least one order** worth **$100 or more** (i.e., `quantity * price >= 100`) in **both** June 2020 **and** July 2020.

**Output:**  
Return a DataFrame with columns `customer_id` and `name`. Each qualifying customer should appear once.

---

## Example

**Input Data**

`customers`:

| customer_id | name    | country |
|-------------|---------|---------|
| 1           | Alice   | USA     |
| 2           | Bob     | UK      |
| 3           | Charlie | Canada  |

`orders`:

| order_id | customer_id | order_date | quantity | price |
|----------|-------------|------------|----------|-------|
| 1        | 1           | 2020-06-10 | 5        | 25    |
| 2        | 1           | 2020-07-15 | 10       | 20    |
| 3        | 2           | 2020-06-20 | 4        | 30    |
| 4        | 2           | 2020-07-05 | 2        | 10    |
| 5        | 3           | 2020-06-01 | 5        | 50    |
| 6        | 3           | 2020-07-20 | 10       | 15    |

**Expected Output:**

| customer_id | name    |
|-------------|---------|
| 1           | Alice   |
| 3           | Charlie |

- **Alice**:  
  - June: `5 * 25 = 125` (≥ 100)  
  - July: `10 * 20 = 200` (≥ 100) → qualifies.
- **Bob**:  
  - June: `4 * 30 = 120` (≥ 100)  
  - July: `2 * 10 = 20` (< 100) → does not qualify.
- **Charlie**:  
  - June: `5 * 50 = 250` (≥ 100)  
  - July: `10 * 15 = 150` (≥ 100) → qualifies.

---


In [4]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType


# Define schema for customers
customers_schema = StructType([
    StructField("customer_id", IntegerType(), True),
    StructField("name", StringType(), True),
    StructField("country", StringType(), True)
])

customers_data = [
    (1, "Alice", "USA"),
    (2, "Bob", "UK"),
    (3, "Charlie", "Canada")
]

customers_df = spark.createDataFrame(customers_data, schema=customers_schema)

# Define schema for orders
orders_schema = StructType([
    StructField("order_id", IntegerType(), True),
    StructField("customer_id", IntegerType(), True),
    StructField("order_date", StringType(), True),
    StructField("quantity", IntegerType(), True),
    StructField("price", IntegerType(), True)
])

orders_data = [
    (1, 1, "2020-06-10", 5, 25),
    (2, 1, "2020-07-15", 10, 20),
    (3, 2, "2020-06-20", 4, 30),
    (4, 2, "2020-07-05", 2, 10),
    (5, 3, "2020-06-01", 5, 50),
    (6, 3, "2020-07-20", 10, 15)
]

orders_df = spark.createDataFrame(orders_data, schema=orders_schema)

customers_df.show()
orders_df.show()

+-----------+-------+-------+
|customer_id|   name|country|
+-----------+-------+-------+
|          1|  Alice|    USA|
|          2|    Bob|     UK|
|          3|Charlie| Canada|
+-----------+-------+-------+

+--------+-----------+----------+--------+-----+
|order_id|customer_id|order_date|quantity|price|
+--------+-----------+----------+--------+-----+
|       1|          1|2020-06-10|       5|   25|
|       2|          1|2020-07-15|      10|   20|
|       3|          2|2020-06-20|       4|   30|
|       4|          2|2020-07-05|       2|   10|
|       5|          3|2020-06-01|       5|   50|
|       6|          3|2020-07-20|      10|   15|
+--------+-----------+----------+--------+-----+



# Using Spark SQL

In [5]:
customers_df.createOrReplaceTempView("customers")
orders_df.createOrReplaceTempView("orders")

In [22]:
spark.sql("""WITH cte AS (
    SELECT
        customer_id,
        MONTH(order_date) AS order_month
    FROM orders
    WHERE YEAR(order_date) = 2020
      AND MONTH(order_date) IN (6, 7)
      AND quantity * price >= 100
    GROUP BY customer_id, MONTH(order_date)
)
SELECT
    c.customer_id,
    c.name  
FROM customers c
JOIN cte t ON c.customer_id = t.customer_id
GROUP BY c.customer_id, c.name
HAVING COUNT(DISTINCT t.order_month) = 2;
3

""").show()

+-----------+-------+
|customer_id|   name|
+-----------+-------+
|          1|  Alice|
|          3|Charlie|
+-----------+-------+



# Using Pyspark

In [24]:
orders_df.printSchema()

root
 |-- order_id: integer (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- order_date: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- price: integer (nullable = true)



In [28]:
orders_df = orders_df.withColumn("order_date",col("order_date").cast("Date"))

In [32]:
from pyspark.sql.functions import col, year, month, expr

# Compute order amount and filter
filtered = orders_df.withColumn("order_amount", col("quantity") * col("price")) \
                    .filter(col("order_amount") >= 100) \
                    .filter((year("order_date") == 2020) & (month("order_date").isin([6, 7])))

  
# Find customers in June and July
june_customers = filtered.filter(month("order_date") == 6).select("customer_id").distinct()
july_customers = filtered.filter(month("order_date") == 7).select("customer_id").distinct()


both_months = june_customers.intersect(july_customers)

result = customers_df.join(both_months, on="customer_id").select("customer_id", "name")
result.show()

+-----------+-------+
|customer_id|   name|
+-----------+-------+
|          1|  Alice|
|          3|Charlie|
+-----------+-------+

